# Implied Volatility Surface — SPY

**Research question:** How does the market price uncertainty across strikes and expiries, and what does the shape of the surface tell us about where the edge is?

This notebook covers:
1. Building the live IV surface from SPY options chain data
2. Fitting the SVI parametrization (Gatheral 2004) per expiry slice
3. Interpreting skew, term structure, and risk reversals
4. Comparing ATM implied vol vs VIX — validating our surface calibration
5. Quantifying the Variance Risk Premium (VRP) as the structural source of edge

---

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from mpl_toolkits.mplot3d import Axes3D
from datetime import date
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

TODAY = date.today().isoformat()
print(f"Running as of: {TODAY}")

## 1. Black-Scholes Greeks — Intuition Check

Before touching market data, let's verify our hand-rolled pricing engine and build intuition for the Greeks. Every formula was derived analytically — no numerical differentiation.

In [ ]:
from src.pricing.black_scholes import all_greeks, price as bs_price
from src.pricing.implied_vol import implied_vol, round_trip_error

# --- Round-trip test ---
S, K, T, r, sigma = 450.0, 450.0, 30/365, 0.05, 0.20
c = bs_price(S, K, T, r, sigma, 'call')
iv_recovered = implied_vol(c, S, K, T, r, 'call')
err = round_trip_error(iv_recovered, c, S, K, T, r, 'call')

print(f"ATM Call Price:      ${c:.4f}")
print(f"IV Round-trip Error: {err:.2e}  (Newton-Raphson converges in ~4 iterations)")
print()

# --- Greeks across a strike range ---
strikes = np.linspace(380, 520, 120)
calls = [all_greeks(S, K_i, T, r, sigma, 'call') for K_i in strikes]
puts  = [all_greeks(S, K_i, T, r, sigma, 'put')  for K_i in strikes]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'Black-Scholes Greeks — ATM SPY-like option  (S={S}, σ={sigma:.0%}, T=30d)', fontsize=13)

greek_specs = [
    ('price',  'Option Price ($)',      'steelblue',  'darkorange'),
    ('delta',  'Delta  dV/dS',          'steelblue',  'darkorange'),
    ('gamma',  'Gamma  d²V/dS²',        'purple',     'purple'),
    ('theta',  'Theta  dV/dt ($/day)',  'green',      'green'),
    ('vega',   'Vega   dV/dσ',          'teal',       'teal'),
    ('vanna',  'Vanna  d²V/(dS·dσ)',    'brown',      'brown'),
]

for ax, (key, label, c_color, p_color) in zip(axes.flat, greek_specs):
    c_vals = [g[key] for g in calls]
    p_vals = [g[key] for g in puts]
    ax.plot(strikes, c_vals, color=c_color, lw=2, label='Call')
    if key not in ('gamma', 'vega', 'vanna'):  # same for C and P
        ax.plot(strikes, p_vals, color=p_color, lw=2, ls='--', alpha=0.7, label='Put')
    ax.axvline(S, color='black', lw=0.8, ls=':', alpha=0.5, label='ATM')
    ax.set_xlabel('Strike')
    ax.set_title(label)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Key observations
- **Gamma** peaks ATM and decays away from the money — this is why ATM straddles have maximum convexity risk for sellers
- **Theta / Gamma relationship**: by the BS PDE, `Θ + ½σ²S²Γ = rV`. Theta and gamma are perfectly inversely related — collecting theta means being short gamma (exposure to large moves)
- **Vanna** (dΔ/dσ) drives skew P&L when spot and vol move together — negative for OTM puts, which is why market makers demand higher IV there

## 2. Live Vol Surface — SPY

Fetch the live options chain, clean it using microstructure filters, solve for implied vols, and fit the SVI parametrization.

In [ ]:
from src.data.fetch import fetch_options_chain, fetch_historical_prices, realized_vol, fetch_vix_history, fetch_vrp_history
from src.vol_surface.surface import VolSurface

TICKER = 'SPY'

print(f"Fetching {TICKER} options chain...")
chain, spot, r = fetch_options_chain(TICKER, as_of=TODAY)
prices = fetch_historical_prices(TICKER, period='2y')

rv_21 = realized_vol(prices, window=21).iloc[-1]
rv_5  = realized_vol(prices, window=5).iloc[-1]

vix_hist = fetch_vix_history(period='2y')
vix_now = float(vix_hist['vix'].iloc[-1])
vix_regime = str(vix_hist['vix_regime'].iloc[-1])

print(f"\n{'='*50}")
print(f"  Spot:       ${spot:.2f}")
print(f"  Risk-free:  {r:.2%}")
print(f"  RV-21d:     {rv_21:.1%}")
print(f"  RV-5d:      {rv_5:.1%}")
print(f"  VIX:        {vix_now:.1f}  [{vix_regime} regime]")
print(f"  VRP:        {(vix_now/100 - rv_21)*100:+.1f} vol pts  (VIX - RV21d)")
print(f"{'='*50}")

In [ ]:
print("Building vol surface (SVI calibration)...")
vs = VolSurface.from_chain(chain, spot=spot, r=r, build_date=TODAY)

ts = vs.atm_term_structure()
print(f"\nCalibrated {len(vs.expiries)} expiry slices\n")
display(ts[['expiry_days', 'atm_iv', 'forward', 'svi_rmse', 'arb_free']].round(4))

## 3. Vol Smiles — Skew Structure

The equity vol smile has a pronounced negative skew (left skew): OTM puts trade at higher IV than OTM calls. This is the **crash risk premium** — investors pay more for downside protection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

expiries_to_plot = sorted(vs.expiries)[:7]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(expiries_to_plot)))

# --- Left: smile by log-moneyness ---
ax = axes[0]
for T, color in zip(expiries_to_plot, colors):
    smile = vs.smile(T)
    ax.plot(smile['log_moneyness'], smile['iv'] * 100, color=color, lw=2, label=f'{T*365:.0f}d')

ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5, label='ATM (k=0)')
ax.set_xlabel('Log-moneyness  k = log(K/F)')
ax.set_ylabel('Implied Vol (%)')
ax.set_title(f'{TICKER} Volatility Smiles  [{TODAY}]')
ax.legend(title='DTE', fontsize=9, ncol=2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

# Annotate the key features
ax.annotate('Left skew:\nCrash risk premium', xy=(-0.2, ax.get_ylim()[1]*0.9),
            fontsize=9, color='darkred', style='italic')

# --- Right: ATM term structure ---
ax = axes[1]
ax.plot(ts['expiry_days'], ts['atm_iv'] * 100, 'o-', color='steelblue', lw=2, ms=7, label='ATM IV (surface)')
ax.axhline(vix_now, color='darkorange', lw=1.5, ls='--', label=f'VIX = {vix_now:.1f}')
ax.axhline(rv_21 * 100, color='green', lw=1.5, ls=':', label=f'RV-21d = {rv_21:.1%}')
ax.set_xlabel('Days to Expiry')
ax.set_ylabel('Implied Vol (%)')
ax.set_title(f'{TICKER} ATM Term Structure  [{TODAY}]')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
plt.show()

## 4. Skew Metrics — 25-Delta Risk Reversal & Butterfly

The two standard vol surface metrics used on trading desks:
- **Risk Reversal** = IV(25Δ put) − IV(25Δ call): measures skew asymmetry (positive = put skew = crash fear)
- **Butterfly** = [IV(25Δ put) + IV(25Δ call)]/2 − IV(50Δ): measures smile curvature (vol-of-vol premium)

In [ ]:
skew_rows = []
for T in sorted(vs.expiries):
    try:
        s = vs.skew(T)
        s['expiry_days'] = round(T * 365)
        s['T'] = T
        skew_rows.append(s)
    except Exception:
        continue

skew_df = pd.DataFrame(skew_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(skew_df['expiry_days'], skew_df['risk_reversal'] * 100, 'o-', color='crimson', lw=2)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('25Δ Risk Reversal by Expiry')
ax.set_xlabel('Days to Expiry')
ax.set_ylabel('RR (vol pts)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

ax = axes[1]
ax.plot(skew_df['expiry_days'], skew_df['butterfly'] * 100, 'o-', color='purple', lw=2)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('25Δ Butterfly (Smile Curvature)')
ax.set_xlabel('Days to Expiry')
ax.set_ylabel('BF (vol pts)')

ax = axes[2]
ax.plot(skew_df['expiry_days'], skew_df['atm_iv'] * 100, 'o-', color='steelblue', lw=2, label='ATM IV')
ax.plot(skew_df['expiry_days'], skew_df['put25_iv'] * 100, 's--', color='red', lw=1.5, alpha=0.7, label='25Δ put IV')
ax.plot(skew_df['expiry_days'], skew_df['call25_iv'] * 100, '^--', color='green', lw=1.5, alpha=0.7, label='25Δ call IV')
ax.set_title('IV by Delta and Expiry')
ax.set_xlabel('Days to Expiry')
ax.set_ylabel('IV (%)')
ax.legend(fontsize=9)

plt.suptitle(f'{TICKER} Skew Structure  [{TODAY}]', fontsize=13)
plt.tight_layout()
plt.show()

display(skew_df[['expiry_days', 'atm_iv', 'put25_iv', 'call25_iv', 'risk_reversal', 'butterfly']].round(4))

## 5. 3D Vol Surface

The full surface visualized — notice the skew deepens for shorter expiries (events have outsized impact near-term) and the overall level is anchored by the VIX.

In [ ]:
grid = vs.surface_grid(k_range=(-0.35, 0.35), n_k=50)

Ts = sorted(grid['T'].unique())
Ks = sorted(grid['log_moneyness'].unique())
Z = grid.pivot(index='T', columns='log_moneyness', values='iv').values * 100

K_mesh, T_mesh = np.meshgrid(Ks, Ts)

fig = plt.figure(figsize=(13, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(K_mesh, T_mesh, Z, cmap='RdYlGn_r', alpha=0.9, edgecolor='none')

ax.set_xlabel('Log-moneyness  log(K/F)', labelpad=10)
ax.set_ylabel('Time to Expiry (years)', labelpad=10)
ax.set_zlabel('Implied Vol (%)', labelpad=10)
ax.set_title(
    f'{TICKER} Implied Volatility Surface\n'
    f'{TODAY} | Spot: ${spot:.2f} | VIX: {vix_now:.1f} | RV-21d: {rv_21:.1%}'
)
fig.colorbar(surf, ax=ax, shrink=0.4, pad=0.1, label='IV (%)')
plt.show()

## 6. VIX vs ATM IV — Surface Validation

The VIX is computed directly from SPX option prices using a model-free formula across the full options strip. Our SPY surface's 30-day ATM IV should closely track VIX (with a small basis due to SPY vs SPX and methodology differences).

If they diverge significantly, it signals a calibration error in our surface.

In [ ]:
# 30-day ATM IV from our surface
T_30d = min(vs.expiries, key=lambda t: abs(t - 30/365))
atm_iv_30d = vs.iv(spot, T_30d)

print(f"Our 30-day ATM IV: {atm_iv_30d:.1%}")
print(f"VIX (SPX 30d IV):  {vix_now/100:.1%}")
print(f"Basis (SPY - VIX): {(atm_iv_30d - vix_now/100)*100:+.1f} vol pts")
print()
print("Note: SPY IV < VIX is normal — SPX is slightly more volatile than SPY,")
print("and VIX includes the full smile (not just ATM).")

## 7. Variance Risk Premium (VRP) — The Edge

The VRP is the systematic spread between implied vol (forward-looking) and realized vol (backward-looking). For the S&P 500:
- **Historical mean VRP: +2 to +4 vol points**
- **Positive >65–70% of trading days**
- **Source:** Option buyers consistently overpay for tail protection

This is the risk premium harvested by delta-hedged short vol strategies.

In [ ]:
vrp_history = fetch_vrp_history(TICKER, period='2y')

print(f"VRP Summary ({TICKER}, 2-year history)")
print(f"  Mean VRP:         {vrp_history['vrp'].mean()*100:+.1f} vol pts")
print(f"  Median VRP:       {vrp_history['vrp'].median()*100:+.1f} vol pts")
print(f"  VRP > 0:          {(vrp_history['vrp'] > 0).mean():.0%} of days")
print(f"  VRP Std Dev:      {vrp_history['vrp'].std()*100:.1f} vol pts")
print(f"  Current VRP:      {vrp_history['vrp'].iloc[-1]*100:+.1f} vol pts")
print()

# Regime breakdown
regime_vrp = vrp_history.groupby('vix_regime')['vrp'].agg(['mean', 'count'])
regime_vrp['mean_vol_pts'] = regime_vrp['mean'] * 100
print("VRP by VIX Regime:")
display(regime_vrp[['count', 'mean_vol_pts']].round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle(f'{TICKER}: Variance Risk Premium Analysis  [{TODAY}]', fontsize=13)

# --- 1. VIX vs RV over time ---
ax = axes[0, 0]
ax.plot(vrp_history.index, vrp_history['iv_proxy'] * 100, label='VIX/100 (implied)', color='darkorange', lw=1.8)
ax.plot(vrp_history.index, vrp_history['rv_21d'] * 100, label='RV-21d (realized)', color='steelblue', lw=1.5)
ax.fill_between(vrp_history.index,
    vrp_history['iv_proxy'] * 100, vrp_history['rv_21d'] * 100,
    where=(vrp_history['iv_proxy'] > vrp_history['rv_21d']),
    alpha=0.2, color='green', label='VRP > 0')
ax.fill_between(vrp_history.index,
    vrp_history['iv_proxy'] * 100, vrp_history['rv_21d'] * 100,
    where=(vrp_history['iv_proxy'] <= vrp_history['rv_21d']),
    alpha=0.3, color='red', label='VRP < 0')
ax.set_title('VIX vs 21-day Realized Vol')
ax.set_ylabel('Vol (%)')
ax.legend(fontsize=8)

# --- 2. VRP time series ---
ax = axes[0, 1]
vrp_pct = vrp_history['vrp'] * 100
ax.bar(vrp_history.index, vrp_pct,
       color=vrp_pct.apply(lambda v: 'seagreen' if v > 0 else 'crimson'),
       alpha=0.7, width=1)
ax.axhline(vrp_pct.mean(), color='black', lw=1.5, ls='--',
           label=f'Mean: {vrp_pct.mean():.1f} vol pts')
ax.axhline(0, color='black', lw=0.8)
ax.set_title('VRP = VIX − 21d RV  (vol points)')
ax.legend(fontsize=9)

# --- 3. VRP distribution ---
ax = axes[1, 0]
ax.hist(vrp_pct, bins=40, color='steelblue', edgecolor='white', alpha=0.8, density=True)
ax.axvline(vrp_pct.mean(), color='red', lw=2, label=f'Mean: {vrp_pct.mean():.1f}')
ax.axvline(0, color='black', lw=1, ls='--')
ax.set_title('VRP Distribution')
ax.set_xlabel('VRP (vol points)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

# --- 4. VRP by regime ---
ax = axes[1, 1]
regime_order = ['low', 'normal', 'elevated', 'crisis']
regime_colors = {'low': 'seagreen', 'normal': 'steelblue', 'elevated': 'darkorange', 'crisis': 'crimson'}
for regime in regime_order:
    subset = vrp_history[vrp_history['vix_regime'] == regime]['vrp'] * 100
    if len(subset) > 5:
        ax.hist(subset, bins=20, alpha=0.6, label=f'{regime} (n={len(subset)})',
                color=regime_colors[regime], density=True)
ax.axvline(0, color='black', lw=1, ls='--')
ax.set_title('VRP Distribution by VIX Regime')
ax.set_xlabel('VRP (vol points)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Key takeaways from the VRP analysis

1. **The VRP is positive on most days** — the structural insurance premium paid by option buyers is real and persistent
2. **The VRP varies by regime** — it's largest in 'low' VIX environments (complacency) and can flip negative in crises (gamma carries more risk than theta collects)
3. **Regime awareness is critical** — a naive "always sell vol" strategy underperforms vs one that sizes down in elevated-VIX regimes

The next notebook (`02_delta_hedge_pnl.ipynb`) will show how these dynamics translate into actual strategy P&L.

## 8. SVI Calibration Quality

Verify the SVI fit — RMSE per expiry and a side-by-side of market vs SVI quotes.

In [ ]:
from src.vol_surface.svi import calibrate_svi, svi_implied_vol, is_arbitrage_free

# Show calibration quality for the 30d slice
T_show = T_30d
result = vs.calibration[T_show]
F = result['forward']

# Market data for this expiry
chain_slice = vs.chain[
    (vs.chain['expiry_years'].round(4) == round(T_show, 4))
].copy()

if len(chain_slice) > 5:
    k_mkt = np.log(chain_slice['strike'].values / F)
    iv_mkt = chain_slice['iv'].values

    k_fine = np.linspace(k_mkt.min(), k_mkt.max(), 200)
    iv_svi = svi_implied_vol(k_fine, T_show, result['params'])

    arb = is_arbitrage_free(result['params'], T_show)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(k_mkt, iv_mkt * 100, color='steelblue', s=40, zorder=5, label='Market quotes')
    ax.plot(k_fine, iv_svi * 100, color='darkorange', lw=2, label=f'SVI fit (RMSE={result["rmse"]*100:.2f} vol pts)')
    ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlabel('Log-moneyness  log(K/F)')
    ax.set_ylabel('Implied Vol (%)')
    ax.set_title(f'{TICKER} SVI Calibration — {T_show*365:.0f}-day Slice  [Arb-free: {arb["arbitrage_free"]}]')
    ax.legend(fontsize=10)

    params = result['params']
    param_str = '  '.join([f'{k}={v:.4f}' for k, v in params.items()])
    ax.set_xlabel(f'Log-moneyness\nSVI params: {param_str}')

    plt.tight_layout()
    plt.show()
else:
    print(f"Insufficient market data for {T_show*365:.0f}d slice (n={len(chain_slice)})")

In [ ]:
# Summary: RMSE and arb-free status for all expiries
calib_summary = []
for T_exp, res in vs.calibration.items():
    arb = res.get('arbitrage_check', {})
    calib_summary.append({
        'expiry_days': round(T_exp * 365),
        'svi_a':    res['params']['a'],
        'svi_b':    res['params']['b'],
        'svi_rho':  res['params']['rho'],
        'svi_m':    res['params']['m'],
        'svi_sigma':res['params']['sigma'],
        'rmse_vol_pts': res.get('rmse', float('nan')) * 100,
        'arb_free': arb.get('arbitrage_free', None),
        'converged': res.get('converged', None),
    })

calib_df = pd.DataFrame(calib_summary).sort_values('expiry_days')
print("SVI Calibration Summary")
display(calib_df.round(4))